# Set up

In [1]:
import scyan as sy
import os
import glob
import anndata
import re
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import scanpy as sc
import scanpy.external as sce


/home/jupyter/envs/libs/scyan/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Global seed set to 0


In [2]:
print(sy.__version__)


1.5.1


In [3]:
# define the working path
panel = "PS1"
data_path='/home/jupyter/projects/pre-ra/flow/raw-data/' + panel + '/labelled-expr/cache/'
fig_path = '/home/jupyter/projects/pre-ra/flow/02-clustering/results/' + panel + '_global_subsample'  + "/""/"
proj_name = 'pre-ra_flow_clustering_' + panel
output_path = '/home/jupyter/projects/pre-ra/flow/02-clustering/data/' +panel +'/'

if not os.path.exists(fig_path):
    os.makedirs(fig_path)
    
if not os.path.exists(output_path):
    os.makedirs(output_path)
    
# define scanpy verbose levels
sc.settings.verbosity = 3
sc.settings.figdir = fig_path
sc.settings.n_jobs = -1

# Helper Functions 

In [4]:
# make a function to find files
def get_filepaths_with_glob(root_path: str, file_regex: str):
    return glob.glob(os.path.join(root_path, file_regex))


def subset_files(file_names, substrings):
    # Initialize an empty list to store the matching file names
    matching_files = []

    # Iterate through each file name
    for file_name in file_names:
        # Check if any substring is present in the file name
        if any(substring in file_name for substring in substrings):
            # If yes, add the file name to the matching list
            matching_files.append(file_name)

    return matching_files

In [5]:
# load data from csv
def read_one(file_path):
    #print(file_path)
    adata = sy.read_csv(file_path, marker_regex='^cd|^hla|tcr|ig|^ccr|klrg|^cx',  exclude_markers=None)
    adata.obs["batch"] = re.findall( 'B\\d\\d\\d', file_path)[0]
    adata.obs["panel"] = re.findall( 'PB1|PT1|PM1|PS1', file_path)[0]
    sample_id = re.findall( 'PB\\d\\d\\d\\d\\d...', file_path)[0]
    adata.obs["sample_id"] = sample_id
    adata.obs["sample.sampleKitGuid"] = "KT" + sample_id[2:7]
    return adata


In [6]:
# Function to check if any substring is present in the file name
def contains_substring(file_name, substrings):
    return any(substring in file_name for substring in substrings)

In [7]:
### UMAP helper plot, define a helper function to set a single column for our UMAP figure legends. 
def one_col_lgd(umap):
    legend = umap.legend(bbox_to_anchor=[1.00, 0.5],
    loc='center left', ncol=1, prop={'size': 6})
    legend.get_frame().set_linewidth(0.0)
    for handle in legend.legendHandles:
        handle.set_sizes([25.0])
    return legend

# Load files

In [8]:
flow_files=get_filepaths_with_glob(data_path, '*/*transform_labelled_expr.csv')


In [10]:
### read in meta data and pull only non duplicate samples to anndata

freq_tbl = pd.read_csv('/home/jupyter/projects/pre-ra/flow/01-qc-reports/data/' + panel+ '/freq_tbl_sample_info_' + panel + '.csv')

In [11]:
freq_tbl.head()

,full_filename,labels,counts,frequency_live,total_counts,sample_id,barcode,l1_labels,filename,panel,...,file.fileType,file.majorVersion,subject.id,subject.biologicalSex,subject.birthYear,subject.ethnicity,subject.partnerCode,subject.race,subject.subjectGuid,cohort.cohortGuid
0,/home//jupyter//projects/pre-ra//flow//raw-dat...,Unknown,29592,0.148508,199262,PB00052-02,3433529e4a9011ee8110ae3437f38a28,unknown,Flow_Cyanno_PS1_PB00052-02_summary_frequency_s...,PS1,...,FlowCytometry-summary-frequency-stats,2,f8424819-e5c7-44fd-b19c-12d8a5dd0771,Female,1963,Non-Hispanic origin,CU,Caucasian,CU1009,CU1
1,/home//jupyter//projects/pre-ra//flow//raw-dat...,basophils,2095,0.010514,199262,PB00052-02,3433535c4a9011ee8110ae3437f38a28,total_myeloid_cells,Flow_Cyanno_PS1_PB00052-02_summary_frequency_s...,PS1,...,FlowCytometry-summary-frequency-stats,2,f8424819-e5c7-44fd-b19c-12d8a5dd0771,Female,1963,Non-Hispanic origin,CU,Caucasian,CU1009,CU1
2,/home//jupyter//projects/pre-ra//flow//raw-dat...,c_dc1,39,0.000196,199262,PB00052-02,343401584a9011ee8110ae3437f38a28,total_myeloid_cells,Flow_Cyanno_PS1_PB00052-02_summary_frequency_s...,PS1,...,FlowCytometry-summary-frequency-stats,2,f8424819-e5c7-44fd-b19c-12d8a5dd0771,Female,1963,Non-Hispanic origin,CU,Caucasian,CU1009,CU1
3,/home//jupyter//projects/pre-ra//flow//raw-dat...,c_dc2,1400,0.007026,199262,PB00052-02,343387b44a9011ee8110ae3437f38a28,total_myeloid_cells,Flow_Cyanno_PS1_PB00052-02_summary_frequency_s...,PS1,...,FlowCytometry-summary-frequency-stats,2,f8424819-e5c7-44fd-b19c-12d8a5dd0771,Female,1963,Non-Hispanic origin,CU,Caucasian,CU1009,CU1
4,/home//jupyter//projects/pre-ra//flow//raw-dat...,cd14pos_cd16pos_monocytes,2650,0.013299,199262,PB00052-02,343359744a9011ee8110ae3437f38a28,total_myeloid_cells,Flow_Cyanno_PS1_PB00052-02_summary_frequency_s...,PS1,...,FlowCytometry-summary-frequency-stats,2,f8424819-e5c7-44fd-b19c-12d8a5dd0771,Female,1963,Non-Hispanic origin,CU,Caucasian,CU1009,CU1


In [12]:
# subset flow files based on non duplicate files in meta data
strings = freq_tbl['sample_id'].unique()

# Subset file_names based on matching substrings
matching_files = [file_name for file_name in flow_files if contains_substring(file_name, strings)]

# Print the result
len(matching_files)
len(strings)

141

In [13]:
flow_files[:5]

['/home/jupyter/projects/pre-ra/flow/raw-data/PS1/labelled-expr/cache/db297d0c-50a1-4f05-9757-bfc2859d75a1/B109_PS1_PB02817-001_live_logical_transform_labelled_expr.csv',
 '/home/jupyter/projects/pre-ra/flow/raw-data/PS1/labelled-expr/cache/8a717d45-fea6-4da7-9f33-28211addb607/B043_PS1_PB00208-01_live_logical_transform_labelled_expr.csv',
 '/home/jupyter/projects/pre-ra/flow/raw-data/PS1/labelled-expr/cache/c3e93bd4-99b9-4013-88d3-72bce8c34014/B140_PS1_PB00076-02_live_logical_transform_labelled_expr.csv',
 '/home/jupyter/projects/pre-ra/flow/raw-data/PS1/labelled-expr/cache/8096d2e2-41ff-4169-b29e-e52c626d4957/B158_PS1_PB04445-001_live_logical_transform_labelled_expr.csv',
 '/home/jupyter/projects/pre-ra/flow/raw-data/PS1/labelled-expr/cache/9d820e65-a233-436e-8ae9-2d22b116f0ca/B104_PS1_PB02136-01_live_logical_transform_labelled_expr.csv']

In [14]:

adata = anndata.concat([read_one(p) for p in matching_files], index_unique="-")

In [15]:
len(adata.obs['sample_id'].unique())

140

In [16]:
adata

AnnData object with n_obs × n_vars = 50861579 × 24
    obs: 'Unnamed: 0', 'sample_id', 'cell_id', 'barcode', 'Time', 'SSC-W', 'SSC-H', 'SSC-A', 'FSC-W', 'FSC-H', 'FSC-A', 'SSC-B-W', 'SSC-B-H', 'SSC-B-A', 'Viability_logicle', 'labels', 'batch', 'panel', 'sample.sampleKitGuid'

In [17]:
adata.obs

,Unnamed: 0,sample_id,cell_id,barcode,Time,SSC-W,SSC-H,SSC-A,FSC-W,FSC-H,FSC-A,SSC-B-W,SSC-B-H,SSC-B-A,Viability_logicle,labels,batch,panel,sample.sampleKitGuid
0-0,0,PB02817-00,1,a1c54a441f6511eda7a03a93708f33ca,0.0,698343.3750,548845.0,6.388038e+05,753859.3125,884942.0,1.111870e+06,695388.4375,524844.0,6.082841e+05,1.428254,cd56dim_nk_cells,B109,PS1,KT02817
1-0,1,PB02817-00,2,a1c54a761f6511eda7a03a93708f33ca,4.0,707237.1250,757549.0,8.929447e+05,747715.6250,638566.0,7.957763e+05,688604.8750,666219.0,7.646028e+05,1.300787,cd56dim_nk_cells,B109,PS1,KT02817
2-0,2,PB02817-00,4,a1c54ac61f6511eda7a03a93708f33ca,10.0,675248.1250,354607.0,3.990795e+05,702365.0625,865534.0,1.013201e+06,639475.4375,361313.0,3.850847e+05,1.658590,memory_treg,B109,PS1,KT02817
3-0,3,PB02817-00,5,a1c54ae41f6511eda7a03a93708f33ca,13.0,689138.1250,504520.0,5.794732e+05,729230.3750,549946.0,6.683956e+05,662501.2500,508409.0,5.613693e+05,1.583088,em_cd4_t_cells,B109,PS1,KT02817
4-0,4,PB02817-00,7,a1c54b341f6511eda7a03a93708f33ca,18.0,697988.8125,599090.0,6.969302e+05,759985.5000,721084.0,9.133556e+05,696755.0625,494779.0,5.745663e+05,1.338426,cd56dim_nk_cells,B109,PS1,KT02817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
362536-161,362536,PB00118-01,448891,b9de66b2200811eda5a202284efa4c79,1296554.0,868966.0625,1493100.0,2.162422e+06,833211.1250,1389701.0,1.929857e+06,866485.6875,1222641.0,1.765668e+06,2.459018,cd14pos_cd16pos_monocytes,B070,PS1,KT00118
362537-161,362537,PB00118-01,448893,b9de670c200811eda5a202284efa4c79,1296561.0,695215.6250,548885.0,6.359891e+05,708255.7500,1084692.0,1.280399e+06,698692.6875,448905.0,5.227444e+05,1.229644,em_cd8_t_cells,B070,PS1,KT00118
362538-161,362538,PB00118-01,448894,b9de6748200811eda5a202284efa4c79,1296563.0,690332.2500,501635.0,5.771580e+05,693805.0000,1133266.0,1.310443e+06,669932.2500,442368.0,4.939277e+05,1.225095,cm_cd4_t_cells,B070,PS1,KT00118
362539-161,362539,PB00118-01,448895,b9de6770200811eda5a202284efa4c79,1296564.0,718765.8750,609197.0,7.297833e+05,687305.0625,1019287.0,1.167602e+06,697612.3125,501837.0,5.834794e+05,1.460785,gd_t_cells,B070,PS1,KT00118


## Merge L1 pop

In [18]:
labels_df = freq_tbl[["labels", "l1_labels"]].drop_duplicates()

In [19]:
adata.obs = adata.obs.join(labels_df.set_index('labels'), on='labels')

In [20]:
adata.obs['l1_labels'].unique()

array(['total_nk_cells', 'total_t_cells', 'total_myeloid_cells',
       'total_b_cells', 'unknown', 'debris', 'Other cells'], dtype=object)

In [21]:
adata

AnnData object with n_obs × n_vars = 50861579 × 24
    obs: 'Unnamed: 0', 'sample_id', 'cell_id', 'barcode', 'Time', 'SSC-W', 'SSC-H', 'SSC-A', 'FSC-W', 'FSC-H', 'FSC-A', 'SSC-B-W', 'SSC-B-H', 'SSC-B-A', 'Viability_logicle', 'labels', 'batch', 'panel', 'sample.sampleKitGuid', 'l1_labels'

In [22]:
# save unscaled adata for subsetting 
adata.write_h5ad(output_path + "adata_preprocess_unscaled_" + panel + ".h5ad")

# QC check

In [ ]:
adata = sc.read_h5ad(output_path + "adata_preprocess_unscaled_" + panel + ".h5ad")

In [ ]:
adata

In [ ]:
# Important: data from the Cyanno pipeline are already logcile tranformed. 
# No need to redo the transformation
adata.X


In [ ]:
## check for NAs in exp array, PM1 has NAns
np.isnan((np.sum(adata.X)))

In [ ]:
adata.to_df().notnull()

# Define highly var markers

In [ ]:
#### ==== DEFINE HIGHLY VAR MARKERS BASED ON PANEL DF ==== ####
# load panel info for pt1
panel_df = pd.read_csv('/home/jupyter/projects/pre-ra/flow/raw-data/AIFI_flow_' + panel + '_panel_breakdown.csv')
gating_antigens = panel_df.loc[panel_df['used_for_clustering']=='Yes', 'antigen']

gating_antigens = [s + '_logicle' for s in gating_antigens]


# set up the variable
adata.var['antigens'] = adata.var.index.str.replace('_logicle', '')
adata.var[['gating_antigens']] = False
adata.var.loc[adata.var.index.isin(gating_antigens),'gating_antigens'] = True
adata.var[['highly_variable']] = adata.var[['gating_antigens']]
adata.var


# Subsample

In [ ]:
#### ==== SUBSAMPLE ==== ####
sc.pp.subsample(adata,n_obs=3500000, random_state = 123)
print(adata)

In [ ]:
## check for NAs in exp array, PM1 has NAns
np.isnan((np.sum(adata.X)))

# Scale

In [ ]:
#### ==== SCALE ==== ####
sy.preprocess.scale(adata)
print(adata.X)

In [ ]:
cell_labels_to_subset_list = adata.obs['l1_labels'].unique()
print(cell_labels_to_subset_list)

print(pd.crosstab(adata.obs['l1_labels'], adata.obs['panel'], margins=True)) 

# Save Scaled

In [ ]:
adata.write_h5ad(output_path  + "adata_preprocess_scaled_global_subsample_3.5mill_" + panel + ".h5ad")


# PCA

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
sce.pp.harmony_integrate(adata, 'batch', adjusted_basis='X_pca_harmony')


In [ ]:
# replace new pca slot
adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']

In [ ]:
sy.tools.umap(adata, markers=gating_antigens)


In [ ]:
p1 = sy.plot.umap(adata, color=['labels','l1_labels', 'batch','panel'], ncols = 2, return_fig = True, size = .5)
p1.set_size_inches(16.5, 16.5)
p1.savefig(fig_path + "global_subsample_3.5mill_" + "_umap_labels_batch_panel_" +panel + ".png",
               dpi=400, bbox_inches='tight')

In [ ]:
p1 = sy.plot.umap(adata, color=['batch'], ncols = 2, return_fig = True, size = .5)
p1.savefig(fig_path + "global_subsample_3.5mill_" + "_umap_batch_" +panel + ".png",
               dpi=400, bbox_inches='tight')

In [ ]:
p1 = sy.plot.umap(adata, color=['l1_labels'], ncols = 2, return_fig = True, size = .5)
p1.savefig(fig_path + "global_subsample_3.5mill_" + "_umap_labels_l1_" +panel + ".png",
               dpi=400, bbox_inches='tight')

In [ ]:
p1 = sy.plot.umap(adata, color=['labels'], ncols = 2, return_fig = True, size = .5)
p1.savefig(fig_path + "global_subsample_3.5mill_" + "_umap_labels_l2_" +panel + ".png",
               dpi=400, bbox_inches='tight')

In [ ]:
adata.obs["Unknown"] = np.nan
adata.obs.loc[adata.obs['labels']=='Unknown',"Unknown"] = "Unknown"
p1=sy.plot.umap(adata, color=['Unknown'], return_fig = True)
p1.savefig(fig_path + "global_subsample_3.5mill_" + "unknown_pop_" +panel + ".png",
               dpi=400, bbox_inches='tight')

In [ ]:
p1=sy.plot.umap(adata, color=adata.var_names.sort_values(),
             ncols=6, show=False, return_fig=True)
p1.set_size_inches(18.5, 18.5)

p1.savefig(fig_path + "global_subsample_3.5mill_" + "_umap_expression_labels_corrected_panel_" +panel + ".png",
               dpi=400, bbox_inches='tight')

In [ ]:
adata

In [ ]:
adata.write_h5ad(output_path + "adata_preprocess_global_subsample_3.5mill_" + panel + ".h5ad")